# Week 74, Day 4: Continuous integration

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w074d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import itertools

matrix = {"python": ["3.10", "3.11", "3.12", "3.13"], "os": ["ubuntu", "windows", "macos"]}
combos = [dict(zip(matrix, values)) for values in itertools.product(*matrix.values())]
print(len(combos), combos[0], combos[-1])
print(sum(1 for c in combos if not (c["os"] == "windows" and c["python"] == "3.10")))

In [ ]:
import itertools

def expand(matrix):
    base = {k: v for k, v in matrix.items() if k not in ("include", "exclude")}
    combos = [dict(zip(base, values)) for values in itertools.product(*base.values())]
    combos = [c for c in combos if not any(all(c.get(k) == v for k, v in ex.items()) for ex in matrix.get("exclude", []))]
    for inc in matrix.get("include", []):
        shared = {k: v for k, v in inc.items() if k in base}
        matched = [c for c in combos if all(c[k] == v for k, v in shared.items())] if shared else []
        if matched and len(shared) == len(base):
            for c in matched:
                c.update(inc)
        else:
            combos.append(dict(inc))
    return combos

matrix = {
    "os": ["ubuntu", "windows"],
    "python": ["3.12", "3.13"],
    "exclude": [{"os": "windows", "python": "3.12"}],
    "include": [{"os": "ubuntu", "python": "3.13", "coverage": True}, {"os": "macos", "python": "3.13"}],
}
for combo in expand(matrix):
    print(combo)

> **Key idea.** CI checks every push and pull request automatically on a clean runner, so nothing unchecked is merged; branch protection makes chosen checks required. A GitHub Actions workflow has triggers (on: push, pull_request, schedule, with branch, tag and path filters), jobs (parallel unless one needs another), runs-on, and steps that either use an action or run commands; ${{ }} inserts matrix values and secrets. A matrix is the product of its lists, minus excludes (which match on all their pairs), plus includes (which extend a fully matching combination or add a new one); fail-fast cancels the rest on a failure. Jobs whose needs fail are skipped. Keep CI fast (cheap checks first, caches keyed on lock files), safe (pinned actions, secrets in settings, least permissions), and reproducible (the same commands locally, with nox or tox).

### Exercise: Expanding a matrix

Write `expand(matrix)` with GitHub's rules: the keys other than `include` and `exclude` form the base matrix (combinations as dicts, in `itertools.product` order); drop every combination matched by an `exclude` entry (a combination matches if it has **all** the entry's key-value pairs); then, for each `include` entry in turn: if it shares **every** base key with the base matrix and matches at least one combination on all of them, add its other keys to every matching combination; otherwise append a copy of it as a new combination. Then `job_names(name, combos)`: names such as `"test (ubuntu, 3.12)"`, listing the combination's values in key order, comma-separated.

In [ ]:
import itertools

def expand(matrix):
    ___

def job_names(name, combos):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_matrix():\n    \"products, excludes and includes\"\n    m = {\"os\": [\"ubuntu\", \"windows\", \"macos\"], \"python\": [\"3.12\", \"3.13\"], \"exclude\": [{\"os\": \"macos\"}, {\"os\": \"windows\", \"python\": \"3.12\"}]}\n    assert expand(m) == [{\"os\": \"ubuntu\", \"python\": \"3.12\"}, {\"os\": \"ubuntu\", \"python\": \"3.13\"}, {\"os\": \"windows\", \"python\": \"3.13\"}]\n    m2 = {\"python\": [\"3.12\", \"3.13\"], \"include\": [{\"python\": \"3.13\", \"coverage\": True}, {\"python\": \"3.14\", \"experimental\": True}]}\n    assert expand(m2) == [{\"python\": \"3.12\"}, {\"python\": \"3.13\", \"coverage\": True}, {\"python\": \"3.14\", \"experimental\": True}]\n    assert len(expand({\"a\": [1, 2, 3], \"b\": [4, 5]})) == 6 and expand({\"include\": [{\"x\": 1}]}) == [{\"x\": 1}]\n    assert job_names(\"test\", expand(m)) == [\"test (ubuntu, 3.12)\", \"test (ubuntu, 3.13)\", \"test (windows, 3.13)\"]")

### Exercise: Which jobs run together?

Write `stages(jobs)`: `jobs` maps each job name to a dict that may have `"needs"` (a job name or a list of them). Return the jobs grouped into stages: stage 1 is every job that needs nothing; each later stage is every remaining job whose needs are all in earlier stages. Sort the names within a stage. Raise `ValueError` for a need that doesn't exist (`"unknown job: NAME"`) or a cycle (`"cycle among: A, B"`, the remaining jobs sorted).

In [ ]:
def stages(jobs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stages():\n    \"a dependency graph in layers\"\n    jobs = {\"lint\": {}, \"types\": {}, \"test\": {\"needs\": [\"lint\", \"types\"]}, \"build\": {\"needs\": \"test\"}, \"docs\": {\"needs\": \"lint\"}, \"publish\": {\"needs\": [\"build\", \"docs\"]}}\n    assert stages(jobs) == [[\"lint\", \"types\"], [\"docs\", \"test\"], [\"build\"], [\"publish\"]]\n    try:\n        stages({\"a\": {\"needs\": \"b\"}, \"b\": {\"needs\": \"a\"}, \"c\": {}})\n        assert False\n    except ValueError as e:\n        assert str(e) == \"cycle among: a, b\"\n    try:\n        stages({\"a\": {\"needs\": \"ghost\"}})\n        assert False\n    except ValueError as e:\n        assert str(e) == \"unknown job: ghost\"")

### Exercise: Success, failure and skipped

Write `run_workflow(jobs, outcome)`: run the jobs in dependency order (as in `stages`); a job whose needs all **succeeded** runs, and its result is `outcome(name)` (`"success"` or `"failure"`); otherwise it's `"skipped"` without calling `outcome`. A job with `"if": "always()"` runs even when its needs failed (but still waits for them). Return `(results, conclusion)`: a dict of job to result, and `"failure"` if any job failed, otherwise `"success"`. Record which jobs `outcome` was called for, in order, in the module-level list `CALLED` (clear it at the start).

In [ ]:
CALLED = []

def run_workflow(jobs, outcome):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_outcomes():\n    \"skipping after a failure\"\n    jobs = {\"lint\": {}, \"test\": {\"needs\": \"lint\"}, \"build\": {\"needs\": \"test\"}, \"notify\": {\"needs\": [\"test\", \"build\"], \"if\": \"always()\"}, \"docs\": {}}\n    results, conclusion = run_workflow(jobs, lambda name: \"failure\" if name == \"test\" else \"success\")\n    assert results == {\"lint\": \"success\", \"docs\": \"success\", \"test\": \"failure\", \"build\": \"skipped\", \"notify\": \"success\"}\n    assert conclusion == \"failure\" and CALLED == [\"docs\", \"lint\", \"test\", \"notify\"]\n    results, conclusion = run_workflow(jobs, lambda name: \"success\")\n    assert conclusion == \"success\" and all(r == \"success\" for r in results.values()) and len(CALLED) == 5")

### Exercise: Does this event start the workflow?

Write `triggers(on, event)`: `on` is a workflow's trigger table (a dict); `event` is a dict with `"type"` (`"push"` or `"pull_request"`), `"branch"` (the pushed branch, or the pull request's target branch), and `"paths"` (the files changed). A trigger with no filters (its value is `None` or `{}`) matches every event of that type. Otherwise every filter given must match: `branches` (a list of glob patterns, with `*` matching anything except `/`, and `**` matching anything), `branches-ignore` (none may match) and `paths` (at least one changed file must match a pattern). Write `glob_match(name, pattern)` for the patterns.

In [ ]:
import re

def glob_match(name, pattern):
    ___

def triggers(on, event):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_triggers():\n    \"branch and path filters\"\n    assert glob_match(\"release/1.0\", \"release/*\") and not glob_match(\"release/1.0/fix\", \"release/*\") and glob_match(\"release/1.0/fix\", \"release/**\")\n    assert glob_match(\"src/shop/cart.py\", \"src/**/*.py\") and glob_match(\"docs/index.md\", \"docs/**\") and not glob_match(\"main\", \"mai\")\n    on = {\"push\": {\"branches\": [\"main\", \"release/*\"]}, \"pull_request\": None}\n    assert triggers(on, {\"type\": \"push\", \"branch\": \"main\", \"paths\": [\"a.py\"]})\n    assert not triggers(on, {\"type\": \"push\", \"branch\": \"feature/x\", \"paths\": [\"a.py\"]})\n    assert triggers(on, {\"type\": \"pull_request\", \"branch\": \"anything\", \"paths\": []})\n    docs = {\"push\": {\"branches-ignore\": [\"dependabot/**\"], \"paths\": [\"docs/**\", \"*.md\"]}}\n    assert triggers(docs, {\"type\": \"push\", \"branch\": \"main\", \"paths\": [\"README.md\"]})\n    assert not triggers(docs, {\"type\": \"push\", \"branch\": \"main\", \"paths\": [\"src/a.py\"]})\n    assert not triggers(docs, {\"type\": \"push\", \"branch\": \"dependabot/pip/rich\", \"paths\": [\"docs/x.md\"]})\n    assert not triggers(docs, {\"type\": \"pull_request\", \"branch\": \"main\", \"paths\": [\"docs/x.md\"]})")

### Exercise: Cache keys

Write `cache_key(prefix, os, python, lock_text)`: `f"{prefix}-{os}-{python}-{h}"`, where `h` is the first 16 hex characters of the SHA-256 of the lock file's text (UTF-8). Then `restore(key, restore_keys, available)`: the cache to use, from the list of `available` cache keys (newest first): the exact `key` if present; otherwise the first available key that **starts with** the first of `restore_keys` that matches anything; otherwise `None`. Return `(chosen, exact)`, where `exact` says whether it was an exact hit.

In [ ]:
import hashlib

def cache_key(prefix, os, python, lock_text):
    ___

def restore(key, restore_keys, available):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cache():\n    \"exact and partial hits\"\n    k = cache_key(\"pip\", \"ubuntu\", \"3.12\", \"rich==13.7.1\\n\")\n    import hashlib\n    assert k == \"pip-ubuntu-3.12-\" + hashlib.sha256(b\"rich==13.7.1\\n\").hexdigest()[:16]\n    assert cache_key(\"pip\", \"ubuntu\", \"3.12\", \"rich==13.7.1\\n\") == k != cache_key(\"pip\", \"ubuntu\", \"3.12\", \"rich==14.0\\n\")\n    available = [\"pip-ubuntu-3.12-aaaa\", \"pip-ubuntu-3.11-bbbb\", \"pip-windows-3.12-cccc\"]\n    assert restore(\"pip-ubuntu-3.12-aaaa\", [], available) == (\"pip-ubuntu-3.12-aaaa\", True)\n    assert restore(\"pip-ubuntu-3.12-zzzz\", [\"pip-ubuntu-3.12-\", \"pip-ubuntu-\"], available) == (\"pip-ubuntu-3.12-aaaa\", False)\n    assert restore(\"pip-macos-3.12-zzzz\", [\"pip-macos-3.12-\", \"pip-ubuntu-\"], available) == (\"pip-ubuntu-3.12-aaaa\", False)\n    assert restore(\"pip-macos-3.12-zzzz\", [\"pip-macos-\"], available) == (None, False)")

### Exercise: Checking a workflow

Write `workflow_problems(wf)`: a list of problems with a workflow (a dict), in this order: `"no triggers"` if `on` is missing or empty; `"no jobs"` if `jobs` is missing or empty; then for each job in order: `"JOB: no runs-on"`, `"JOB: no steps"`, `"JOB: needs unknown job NAME"` (for each), and for each step (counting from 1): `"JOB step N: needs uses or run"` (it has neither), `"JOB step N: both uses and run"`, and `"JOB step N: unpinned action NAME"` for a `uses` value with no `@` version.

In [ ]:
def workflow_problems(wf):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_workflow():\n    \"a workflow linter\"\n    good = {\"on\": {\"push\": None}, \"jobs\": {\"test\": {\"runs-on\": \"ubuntu-latest\", \"steps\": [{\"uses\": \"actions/checkout@v4\"}, {\"run\": \"pytest\"}]}}}\n    assert workflow_problems(good) == []\n    bad = {\"on\": {\"push\": None}, \"jobs\": {\n        \"lint\": {\"steps\": [{\"uses\": \"actions/checkout\"}, {\"name\": \"nothing\"}]},\n        \"test\": {\"runs-on\": \"ubuntu-latest\", \"needs\": [\"lint\", \"build\"], \"steps\": [{\"uses\": \"x/y@v1\", \"run\": \"ls\"}]},\n        \"empty\": {\"runs-on\": \"ubuntu-latest\"},\n    }}\n    assert workflow_problems(bad) == [\n        \"lint: no runs-on\", \"lint step 1: unpinned action actions/checkout\", \"lint step 2: needs uses or run\",\n        \"test: needs unknown job build\", \"test step 1: both uses and run\", \"empty: no steps\",\n    ]\n    assert workflow_problems({}) == [\"no triggers\", \"no jobs\"]")

### Bug hunt: An exclude that removed too much

The exclude should drop only Windows with Python 3.12, but the check drops any combination that matches **either** pair, so this prints `1` (only Ubuntu with 3.13 is left). A combination should be excluded only if it matches **all** the entry's pairs, so this prints `3`.

In [ ]:
import itertools

combos = [dict(zip(["os", "python"], v)) for v in itertools.product(["ubuntu", "windows"], ["3.12", "3.13"])]
exclude = {"os": "windows", "python": "3.12"}
kept = [c for c in combos if not any(c[k] == v for k, v in exclude.items())]
print(len(kept))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"all pairs must match\"\n    expect_output(\"3\")")

### Bug hunt: A branch pattern compared as text

The trigger compares the branch with each pattern using `==`, so the pattern `release/*` never matches `release/2.0` and this prints `False`. Match with `fnmatch.fnmatchcase` (close enough for single-level patterns), so it prints `True`.

In [ ]:
import fnmatch

branches = ["main", "release/*"]
pushed = "release/2.0"
print(any(pushed == pattern for pattern in branches))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"glob patterns\"\n    expect_output(\"True\")")

### Bug hunt: Jobs that ran in the wrong order

The runner sorts jobs by name and ignores `needs`, so `build` runs before the `test` it needs, and this prints `['build', 'lint', 'test']`. Run a job only after everything it needs, so it prints `['lint', 'test', 'build']`.

In [ ]:
jobs = {"build": {"needs": ["test"]}, "lint": {}, "test": {"needs": ["lint"]}}
order = sorted(jobs)
print(order)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"dependency order\"\n    expect_output(\"['lint', 'test', 'build']\")")

### Puzzle: A test job

Arrange the lines of a GitHub Actions job that checks out the code, installs Python from the matrix, installs the project and runs the tests. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # combinations in a matrix of 3 operating systems and 4 Python versions, with no includes or excludes
q = ___   # what happens to a job whose needed job failed (without "if: always()"): "skipped" or "runs"
r = ___   # True or False: a secret should be written in the workflow file so CI can read it
s = ___   # the default value of fail-fast in a GitHub Actions matrix: True or False
t = ___   # what a dependency cache key should include so it changes when dependencies do: "the date" or "a hash of the lock file"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 12 and q == \"skipped\" and r is False, \"p, q and r\"\n    assert s is True and t == \"a hash of the lock file\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a small CI engine

Write `run_ci(workflow, execute)`. Each job may have a `"strategy"` with a `"matrix"` (expanded with `expand`, given) and `"fail-fast"` (default `True`); a job without a matrix runs once. Instances are named as in the practice (`"test (ubuntu, 3.12)"`), or just the job name with no matrix. Process jobs stage by stage (sorted within a stage, as in `stages`, given), and a job's instances in matrix order. If any job it needs **didn't fully succeed**, every instance of the job is `"skipped"`. Otherwise call `execute(instance_name, combo)` (with `{}` for no matrix), which returns `True` for success. With fail-fast on, once an instance of a job fails, its later instances are `"cancelled"` without being executed. A job **succeeded** if every instance succeeded. Return `(results, conclusion)`: results maps each instance name to `"success"`, `"failure"`, `"cancelled"` or `"skipped"`, and the conclusion is `"failure"` if any instance failed or was cancelled, else `"success"`.

In [ ]:
import itertools

def expand(matrix):
    base = {k: v for k, v in matrix.items() if k not in ("include", "exclude")}
    combos = [dict(zip(base, values)) for values in itertools.product(*base.values())] if base else []
    combos = [c for c in combos if not any(all(c.get(k) == v for k, v in ex.items()) for ex in matrix.get("exclude", []))]
    for inc in matrix.get("include", []):
        matched = [c for c in combos if all(c[k] == inc[k] for k in base)] if base and all(k in inc for k in base) else []
        if matched:
            for c in matched:
                c.update({k: v for k, v in inc.items() if k not in base})
        else:
            combos.append(dict(inc))
    return combos

def stages(jobs):
    def needs(job):
        n = job.get("needs", [])
        return [n] if isinstance(n, str) else list(n)
    done, remaining, result = set(), set(jobs), []
    while remaining:
        stage = sorted(j for j in remaining if all(n in done for n in needs(jobs[j])))
        if not stage:
            raise ValueError("cycle")
        result.append(stage)
        done.update(stage)
        remaining.difference_update(stage)
    return result

def run_ci(workflow, execute):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_engine():\n    \"matrices, fail-fast and skipped jobs\"\n    wf = {\"on\": {\"push\": None}, \"jobs\": {\n        \"lint\": {\"runs-on\": \"ubuntu\"},\n        \"test\": {\"needs\": \"lint\", \"strategy\": {\"matrix\": {\"os\": [\"ubuntu\", \"windows\"], \"py\": [\"3.12\", \"3.13\"]}}},\n        \"build\": {\"needs\": \"test\"},\n    }}\n    calls = []\n    def execute(name, combo):\n        calls.append(name)\n        return not (combo.get(\"os\") == \"windows\" and combo.get(\"py\") == \"3.12\")\n    results, conclusion = run_ci(wf, execute)\n    assert results == {\"lint\": \"success\", \"test (ubuntu, 3.12)\": \"success\", \"test (ubuntu, 3.13)\": \"success\",\n                       \"test (windows, 3.12)\": \"failure\", \"test (windows, 3.13)\": \"cancelled\", \"build\": \"skipped\"}\n    assert conclusion == \"failure\" and calls == [\"lint\", \"test (ubuntu, 3.12)\", \"test (ubuntu, 3.13)\", \"test (windows, 3.12)\"]\n    wf[\"jobs\"][\"test\"][\"strategy\"][\"fail-fast\"] = False\n    results, _ = run_ci(wf, execute)\n    assert results[\"test (windows, 3.13)\"] == \"success\" and results[\"build\"] == \"skipped\"\n    results, conclusion = run_ci(wf, lambda name, combo: True)\n    assert conclusion == \"success\" and results[\"build\"] == \"success\" and len(results) == 6")

## Warm-up

Two short lists in a workflow file become twelve separate test runs, on three operating systems, all at once. That's the CI **matrix**, and it's how a small project tests every Python version it claims to support (in `requires-python`, Week 73 Day 2) without anyone owning twelve machines.

## What CI is for

**Continuous integration** means merging everyone's work into the main branch often, in small pieces, with every change **checked automatically** first. A CI service watches the repository; on each push or pull request it starts a fresh virtual machine (a **runner**), checks out the code, and runs the project's checks: formatting, linting, type checking, tests (with coverage), building the package. The result is reported on the commit and the pull request.

What it buys you:

- **Nothing is merged unchecked.** "It works on my machine" stops meaning anything: the checks run on a clean machine, from what's in the repository, so a forgotten file or an uncommitted dependency fails at once.
- **Problems are found while they're small**: in the pull request that caused them, not weeks later.
- **Reviewers don't check what machines can** (Day 2): by the time a person reads the change, it's formatted, lint-free and passing.
- **Branch protection** can make chosen checks **required**: the merge button stays disabled until they pass.

Related terms: **continuous delivery** keeps the main branch always ready to release; **continuous deployment** releases every change that passes, automatically. A typical Python project's CI is a pipeline: quick checks (format, lint) first, then the test matrix, then building the distributions, and, for a tag, publishing them (with trusted publishing, Week 73 Day 3).

## A workflow, line by line

GitHub Actions workflows are YAML files in `.github/workflows/`. A typical one for a Python library:

```yaml
name: CI

on:
  push:
    branches: [main]
  pull_request:

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install ruff
      - run: ruff check . && ruff format --check .

  test:
    needs: lint
    runs-on: ${{ matrix.os }}
    strategy:
      fail-fast: false
      matrix:
        os: [ubuntu-latest, windows-latest]
        python-version: ["3.10", "3.11", "3.12", "3.13"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip
      - run: pip install -e ".[dev]"
      - run: pytest --cov=textkit --cov-report=term-missing
```

- **`on`** lists the **triggers**: here, pushes to `main` and every pull request. Filters narrow them: `branches`, `tags` and `paths` (run only when matching files change), with `*` and `**` wildcards. Other triggers include `schedule` (a cron time) and `workflow_dispatch` (a button).
- **`jobs`** run **in parallel** by default, each on its own fresh runner. **`needs`** makes a job wait for others, and skips it if they fail.
- **`runs-on`** chooses the machine: `ubuntu-latest`, `windows-latest`, `macos-latest`.
- **`steps`** run in order; each either **`uses`** an action (a reusable, versioned step, such as checking out the code or installing Python) or **`run`s** shell commands. A step that exits with a non-zero status fails the job.
- **`${{ ... }}`** expressions insert values: matrix values, secrets (`${{ secrets.NAME }}`, never written in the file), and context such as the branch name.

YAML isn't in Python's standard library, so today's exercises treat workflows as the dictionaries a YAML parser would produce: the same structure, with Python syntax.

## Matrices and dependencies

A **matrix** runs one job once per combination of its values, the Cartesian product of Week 59's `itertools.product`. Two keys adjust it:

- **`exclude`** lists combinations to drop. An exclude entry matches any combination containing **all** of its key-value pairs, so `{"os": "windows-latest", "python-version": "3.10"}` drops one combination, and `{"os": "macos-latest"}` drops every macOS one.
- **`include`** adds to the matrix. If an include entry's values match an existing combination on every key they share with the matrix, it **adds** its extra keys to that combination (to set an extra variable for one case); otherwise it's appended as a **new** combination.

`fail-fast` (on by default) cancels the rest of the matrix as soon as one combination fails, which saves time but hides how widespread a failure is; `fail-fast: false` lets them all finish.

Jobs with **`needs`** form a dependency graph (a DAG, as in Week 50): a job runs when all the jobs it needs have **succeeded**; if one of them failed or was skipped, it's **skipped** too. The workflow **fails** if any job failed.

## Fast, safe and reproducible

CI only helps if people trust it and don't wait long for it:

- **Order by speed**: lint and formatting checks take seconds and catch many problems, so run them first, or as a job the slow ones `need`.
- **Cache** downloaded packages between runs, keyed on a **hash of the lock file** (Week 73 Day 5): when dependencies change, the key changes and the cache is rebuilt; otherwise installation takes seconds. `setup-python`'s `cache: pip` does this for you.
- **Pin your actions** to a version (`actions/checkout@v4`, or a full commit hash for the strictest security), so a change to someone else's action can't silently change your build.
- **Secrets** live in the repository's or organisation's settings, are passed in as `${{ secrets.NAME }}`, are masked in logs, and aren't given to workflows running on pull requests from forks. Prefer short-lived credentials (trusted publishing) over long-lived ones. Give each workflow only the `permissions` it needs.
- **Flaky tests** (Week 72 Day 5) make people ignore red crosses. Fix them at once; never "re-run until green".
- **Reproduce locally.** Put the commands in one place the developer can also run: a tool such as **nox** or **tox** (which create environments and run sessions such as `nox -s tests-3.12`), or a `Makefile`, and have CI call the same commands. Then a CI failure is something you can debug on your own machine.

## Practice

## On your own computer

Give a project of yours a CI workflow and a matching local command:

```bash
mkdir -p .github/workflows && pip install nox
```

```python
# noxfile.py: the same sessions locally and in CI ("nox -s tests" or "nox -s lint")
import nox

@nox.session(python=["3.12", "3.13"])
def tests(session):
    session.install("-e", ".[dev]")
    session.run("pytest", "--cov")

@nox.session
def lint(session):
    session.install("ruff")
    session.run("ruff", "check", ".")
    session.run("ruff", "format", "--check", ".")
```

Then write `.github/workflows/ci.yml` as in today's example, with steps that run `nox -s lint` and `nox -s tests`, push it, and watch the checks appear on your next pull request. In the repository settings, add a branch protection rule for `main` that requires them.

## Quiz

## Daily challenge

## Before you go

Look at the `.github/workflows` folder of a Python project you use: find its triggers, its matrix and the job that publishes releases. Tomorrow: **pre-commit hooks**, which run the quick checks on your own machine before a commit is even made.